In [1]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load notes + the same embedding model from Phase 4
with open("../data/processed/fa_notes.json") as f:
    notes = json.load(f)
texts = [n["fa_note"] for n in notes]

model = SentenceTransformer("all-MiniLM-L6-v2")
note_embeddings = model.encode(texts)   # our knowledge base, embedded
print("Knowledge base:", note_embeddings.shape)

def semantic_search(query, top_k=3):
    q_emb = model.encode([query])                       # embed the question
    sims = cosine_similarity(q_emb, note_embeddings)[0]  # similarity to every note
    top_idx = sims.argsort()[::-1][:top_k]               # highest-similarity notes
    return [(notes[i], float(sims[i])) for i in top_idx]

# Test it
query = "failures caused by temperature problems in the thermal step"
results = semantic_search(query)
for note, score in results:
    print(f"\n[score {score:.3f}] wafer {note['wafer_idx']}")
    print(note["fa_note"][:160], "...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base: (104, 384)

[score 0.458] wafer 277
Wafer 277 exhibited parametric drift at final electrical test. The moderate deviation was concentrated in sensor 562 (-2.2σ), sensor 55 (+2.1σ), sensor 64 (+2.0 ...

[score 0.456] wafer 1303
Wafer 1303 failed functional verification with anomalous measurements. The moderate deviation was concentrated in sensor 473 (+4.9σ), sensor 474 (+3.8σ), sensor ...

[score 0.451] wafer 231
Wafer 231 showed out-of-spec readings during end-of-line testing. The moderate deviation was concentrated in sensor 468 (+3.2σ), sensor 28 (-2.3σ), sensor 562 ( ...


In [2]:
print("\n\n=== Sensor-specific query ===")
for note, score in semantic_search("problems with sensor 567"):
    print(f"[score {score:.3f}] wafer {note['wafer_idx']}: {note['fa_note'][:120]}...")



=== Sensor-specific query ===
[score 0.559] wafer 583: Wafer 583 registered abnormal process signatures flagged at test. The moderate deviation was concentrated in sensor 40 (...
[score 0.505] wafer 58: Wafer 58 exhibited parametric drift at final electrical test. The severe deviation was concentrated in sensor 164 (+7.5σ...
[score 0.502] wafer 1325: Wafer 1325 failed functional verification with anomalous measurements. The moderate deviation was concentrated in sensor...


In [3]:
from rank_bm25 import BM25Okapi
import numpy as np

# Install if needed: pip install rank-bm25
# Tokenize notes for keyword search
tokenized = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized)

def hybrid_search(query, top_k=3, alpha=0.5):
    # --- semantic component ---
    q_emb = model.encode([query])
    sem = cosine_similarity(q_emb, note_embeddings)[0]
    sem_norm = (sem - sem.min()) / (sem.max() - sem.min() + 1e-9)

    # --- keyword component ---
    kw = np.array(bm25.get_scores(query.lower().split()))
    kw_norm = (kw - kw.min()) / (kw.max() - kw.min() + 1e-9)

    # --- fuse: weighted blend ---
    combined = alpha * sem_norm + (1 - alpha) * kw_norm
    top_idx = combined.argsort()[::-1][:top_k]
    return [(notes[i], float(combined[i]), float(sem_norm[i]), float(kw_norm[i]))
            for i in top_idx]

print("=== Sensor 567 query (hybrid) ===")
for note, comb, s, k in hybrid_search("problems with sensor 567"):
    hit = "567" in note["fa_note"]
    print(f"[combined {comb:.2f} | sem {s:.2f} | kw {k:.2f}] wafer {note['wafer_idx']} | contains '567': {hit}")
    print("  ", note["fa_note"][:110], "...")

=== Sensor 567 query (hybrid) ===
[combined 0.79 | sem 0.61 | kw 0.98] wafer 1365 | contains '567': True
   Wafer 1365 registered abnormal process signatures flagged at test. The severe deviation was concentrated in se ...
[combined 0.76 | sem 0.51 | kw 1.00] wafer 1242 | contains '567': True
   Wafer 1242 showed out-of-spec readings during end-of-line testing. The severe deviation was concentrated in se ...
[combined 0.64 | sem 0.30 | kw 0.99] wafer 2 | contains '567': True
   Wafer 2 exhibited parametric drift at final electrical test. The severe deviation was concentrated in sensor 5 ...


In [ ]:
import os
from dotenv import load_dotenv


def build_rag_prompt(query, retrieved):
    context = "\n\n".join(
        f"[FA Note - Wafer {n['wafer_idx']}]\n{n['fa_note']}"
        for n, *_ in retrieved
    )
    return f"""You are a semiconductor failure-analysis assistant. Answer the engineer's question using ONLY the failure-analysis notes provided as context. Cite the wafer numbers you draw from. If the context doesn't contain the answer, say so.

CONTEXT:
{context}

ENGINEER'S QUESTION: {query}

ANSWER:"""

def rag_answer(query, top_k=3, alpha=0.5):
    retrieved = hybrid_search(query, top_k=top_k, alpha=alpha)
    prompt = build_rag_prompt(query, retrieved)

    # Show the grounded prompt (this is the offline, demonstrable artifact)
    print("=== RETRIEVED CONTEXT ASSEMBLED INTO PROMPT ===")
    print(prompt)
    print("\n" + "="*50)

    # Generation step — runs when API credits are available
    load_dotenv()
    try:
        from anthropic import Anthropic
        client = Anthropic()
        msg = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=400,
            messages=[{"role": "user", "content": prompt}],
        )
        print("\n=== LLM ANSWER ===")
        print(msg.content[0].text)
    except Exception as e:
        print(f"\n[Generation skipped — {type(e).__name__}: add API credits to enable]")
        print("Retrieval + prompt assembly above is fully functional.")

rag_answer("What failure modes involve the most severe sensor deviations?")

ModuleNotFoundError: No module named 'src'